# Drones
Voici les solutions proposees pour les drones. Nous avons decide de prendre l'algorithme du probleme du facteur chinois.

## 1. Trouver le meilleur chemin

### I. Initialisation


In [1]:
INF = float("inf")

import math
from collections import defaultdict
import itertools
import numpy as np
import random as rd
import time
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


### II. Matrice d'adjacence

In [2]:
adj_matrix = [
    [0, 2, 2, 1],  # A
    [2, 0, 3, 0],  # B
    [2, 3, 0, 4],  # C
    [1, 0, 4, 0],  # D
]
nodes = ['A', 'B', 'C', 'D']

### III. Matrice en dictionnaire
À partir de la matrice d'adjacence donnée en entrée, on construit une structure de graphe exploitable (dictionnaire). Cela permet de mieux représenter les connexions entre les sommets. Chaque sommet est lié aux autres par des arêtes portant un poids, représentant par exemple la distance ou le temps de parcours.

In [3]:
def matrix_to_graph(matrix, nodes):
    graph = defaultdict(dict)
    for i in range(len(matrix)):
        for j in range(i, len(matrix)):
            if matrix[i][j] > 0:
                u, v = nodes[i], nodes[j]
                graph[u][v] = matrix[i][j]
                graph[v][u] = matrix[i][j]
    return graph



### IV. Détecter sommets de degrés impairs
On parcourt tous les sommets du graphe pour calculer leur degré. Un graphe est eulérien seulement si tous ses sommets ont un degré pair. Dans cette étape, on repère les sommets qui ont un degré impair. Ce sont eux qu’il faudra relier par des chemins supplémentaires pour équilibrer le graphe.

In [4]:
def get_odd_nodes(graph):
    return [node for node in graph if len(graph[node]) % 2 != 0]

### V. Dijkstra
Pour chaque sommet de degré impair, on calcule le chemin le plus court vers tous les autres sommets (y compris les autres sommets impairs), en utilisant un algorithme comme Dijkstra. On obtient ainsi une carte des distances minimales entre chaque paire de sommets impairs, ainsi que les chemins exacts permettant de les relier efficacement.

In [5]:
def dijkstra(graph, start):
    dist = {v: math.inf for v in graph}
    prev = {v: None for v in graph}
    dist[start] = 0
    q = set(graph.keys())

    while q:
        u = min(q, key=lambda v: dist[v])
        q.remove(u)
        for v in graph[u]:
            alt = dist[u] + graph[u][v]
            if alt < dist[v]:
                dist[v] = alt
                prev[v] = u

    return dist, prev

def shortest_path(prev, target):
    path = []
    while target:
        path.append(target)
        target = prev[target]
    return path[::-1]

### VI. Couplage parfait min
Une fois les distances connues entre les sommets impairs, on cherche à former des paires de sommets impairs de façon à minimiser la somme des distances des chemins qui les relient. Cela revient à résoudre un problème de couplage parfait de poids minimal, c’est-à-dire former des paires sans chevauchement qui coûtent le moins cher possible.

In [6]:
def get_min_matching(odd_nodes, all_distances):
    best = None
    min_cost = INF
    print(itertools.permutations(odd_nodes))
    for pairs in itertools.permutations(odd_nodes):
        if len(pairs) % 2 != 0:
            continue
        used = set()
        cost = 0
        match = []
        for i in range(0, len(pairs), 2):
            u, v = pairs[i], pairs[i+1]
            if u in used or v in used:
                break
            cost += all_distances[u][v]
            match.append((u, v))
            used.add(u)
            used.add(v)
        if len(used) == len(odd_nodes) and cost < min_cost:
            min_cost = cost
            best = match
    return best

### VII. Ajouter les arêtes
Pour chaque paire de sommets impairs trouvée à l'étape précédente, on duplique les arêtes du chemin le plus court entre eux. Cela revient à ajouter temporairement ces trajets au graphe. En conséquence, tous les sommets deviennent de degré pair, rendant le graphe eulérien. Ces duplications simulent le fait que le facteur repassera par certaines rues deux fois.

In [7]:
def add_duplicate_edges(graph, matching, paths):
    for u, v in matching:
        path = paths[(u, v)]
        for i in range(len(path) - 1):
            a, b = path[i], path[i+1]
            if b in graph[a]:
                graph[a][b] += 0.001  # duplication logique
                graph[b][a] += 0.001
            else:
                graph[a][b] = graph[b][a] = 1


### VIII. Construction circuit eulerien (HIERHOLZER)
Une fois le graphe rendu eulérien, on peut appliquer un algorithme (Hierholzer) pour trouver un circuit eulérien : un chemin fermé qui passe exactement une fois par chaque arête (y compris les doublées). Ce chemin représente la tournée optimale du postier : il couvre tout le réseau sans répétitions inutiles et avec un coût total minimal.

In [8]:
def find_eulerian_tour(graph):
    g = {u: list(v.items()) for u, v in graph.items()}
    circuit = []
    stack = [next(iter(g))]
    while stack:
        u = stack[-1]
        if g[u]:
            v, w = g[u].pop()
            g[v].remove((u, w))
            stack.append(v)
        else:
            circuit.append(stack.pop())
    return circuit[::-1]

### IX. Execution

In [9]:
graph = matrix_to_graph(adj_matrix, nodes)
odd_nodes = get_odd_nodes(graph)

all_dist = {}
all_paths = {}
for u in odd_nodes:
    dist, prev = dijkstra(graph, u)
    for v in odd_nodes:
        if u != v:
            all_dist.setdefault(u, {})[v] = dist[v]
            all_paths[(u, v)] = shortest_path(prev, v)
#print("all_dist: ", all_dist)
#print("all_paths: ", all_paths) 

matching = get_min_matching(odd_nodes, all_dist)

add_duplicate_edges(graph, matching, all_paths)

circuit = find_eulerian_tour(graph)
print("Circuit eulerien trouve :")
print(" → ".join(circuit))

Circuit eulerien trouve :
A → D → C → B → A → C


### X. Randomization

In [10]:
def gen_random_math(n):
    l = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            if (i != j and rd.randint(0, 4) == 1):
                    r = rd.randint(1, 100)
                    l[i][j] = r
                    l[j][i] = r
    for i in range(n):
        if (np.sum(l[i]) == 0):
            coord = rd.randint(0,n)
            if (coord == i):
                coord = (i+1)%n
                r = rd.randint(1, 100), rd.randint(0,1)
                l[i][coord] = r
                l[coord][i] = r
    return l
    
n = 10
adj_matrix = gen_random_math(n)
adj_matrix

array([[ 0, 40,  0,  0, 88, 76,  0,  0,  0,  0],
       [40,  0,  0, 23, 23, 14,  0,  0,  0,  0],
       [ 0,  0,  0, 69,  0,  0,  0,  0,  0,  0],
       [ 0, 23, 69,  0,  0,  0,  0, 68, 20,  0],
       [88, 23,  0,  0,  0,  0, 65,  0,  0,  0],
       [76, 14,  0,  0,  0,  0,  0,  8,  0,  0],
       [ 0,  0,  0,  0, 65,  0,  0, 70,  0,  0],
       [ 0,  0,  0, 68,  0,  8, 70,  0,  0, 96],
       [ 0,  0,  0, 20,  0,  0,  0,  0,  0,  3],
       [ 0,  0,  0,  0,  0,  0,  0, 96,  3,  0]])

In [11]:
nodes = [str(i) for i in range(n)]
graph = matrix_to_graph(adj_matrix, nodes)

odd_nodes = get_odd_nodes(graph)

all_dist = {}
all_paths = {}
for u in odd_nodes:
    dist, prev = dijkstra(graph, u)
    for v in odd_nodes:
        if u != v:
            all_dist.setdefault(u, {})[v] = dist[v]
            all_paths[(u, v)] = shortest_path(prev, v)
#print("all_dist: ", all_dist)
#print("all_paths: ", all_paths) 

matching = get_min_matching(odd_nodes, all_dist)

add_duplicate_edges(graph, matching, all_paths)

circuit = find_eulerian_tour(graph)
print("Circuit eulerien trouve :")
print(" → ".join(circuit))

Circuit eulerien trouve :
0 → 5 → 7 → 9 → 8 → 3 → 7 → 6 → 4 → 1 → 0 → 4 → 3 → 2 → 5


## 2. Detection de la neige

### I. Creer une ville aleatoirement avec de la neige

In [12]:
def est_connexe(matrice_adjacence):
    n = len(matrice_adjacence)
    visites = [False] * n

    def dfs(sommet):
        visites[sommet] = True
        for voisin in range(n):
            if matrice_adjacence[sommet][voisin] and not visites[voisin]:
                dfs(voisin)
    dfs(0)
    return all(visites)

def without_snow(l):
    copy = np.array(l)
    return copy[:,:,0]

def gen_random_city(n):
    connexe = False
    while not connexe:
        l = [[(0,0) for i in range(n)] for j in range(n)]
        for i in range(n):
            for j in range(n):
                if (i != j and rd.randint(0, 10) == 1):
                        r = rd.randint(1, 100), rd.randint(0,1)
                        l[i][j] = r
                        l[j][i] = r
        for i in range(n):
            if (np.sum(l[i]) == 0):
                coord = rd.randint(0,n)
                if (coord == i):
                    coord = (i+1)%n
                    r = rd.randint(1, 100), rd.randint(0,1)
                    l[i][coord] = r
                    l[coord][i] = r
        connexe = est_connexe(without_snow(l))
    return l

### II. Trouver le chemin
On reprend les fonctions precedentes pour determiner le chemin eulerien.

In [13]:
n = 17

adj_matrix_snow = gen_random_city(n)
adj_matrix = without_snow(adj_matrix_snow)
print(adj_matrix_snow)

print(est_connexe(adj_matrix))

[[(0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (20, 1), (17, 1), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (21, 0), (0, 0)], [(0, 0), (0, 0), (0, 0), (24, 0), (46, 1), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (10, 1), (0, 0)], [(0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (4, 0), (54, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (96, 1), (0, 0), (0, 0), (0, 0)], [(0, 0), (24, 0), (0, 0), (0, 0), (62, 1), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (81, 0), (0, 0), (0, 0), (0, 0), (0, 0)], [(0, 0), (46, 1), (0, 0), (62, 1), (0, 0), (0, 0), (0, 0), (79, 0), (0, 0), (0, 0), (0, 0), (0, 0), (87, 0), (0, 0), (36, 0), (86, 1), (24, 1)], [(0, 0), (0, 0), (4, 0), (0, 0), (0, 0), (0, 0), (26, 1), (95, 1), (34, 0), (23, 1), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0)], [(0, 0), (0, 0), (54, 0), (0, 0), (0, 0), (26, 1), (0, 0), (0, 0), (56, 1), (51, 1), (0, 0), (43, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0)], [(20, 

In [14]:
start = time.time()

nodes = [str(i) for i in range(n)]
graph = matrix_to_graph(adj_matrix, nodes)

odd_nodes = get_odd_nodes(graph)

all_dist = {}
all_paths = {}
for u in odd_nodes:
    dist, prev = dijkstra(graph, u)
    for v in odd_nodes:
        if u != v:
            all_dist.setdefault(u, {})[v] = dist[v]
            all_paths[(u, v)] = shortest_path(prev, v)
#print("all_dist: ", all_dist)
#print("all_paths: ", all_paths) 

matching = get_min_matching(odd_nodes, all_dist)

add_duplicate_edges(graph, matching, all_paths)

circuit = find_eulerian_tour(graph)

end = time.time()
print("Circuit eulerien trouve :")
print(" → ".join(circuit))
print("Temps total :", end - start, "secondes.")

KeyboardInterrupt: 

### III. Lister les chemins enneiges

In [ ]:
def snow_roads(chem_eul, matrix):
    L = []
    for i in range(len(chem_eul) - 1):
        if (matrix[int(chem_eul[i])][int(chem_eul[i+1])][1] == 1):
            L.append((int(chem_eul[i]), int(chem_eul[i+1]), 0))
    return L

In [ ]:
res = snow_roads(circuit,adj_matrix_snow)

print("Les chemins enneiges sont :", res)

In [ ]:
def display_path(l):
    if (len(l) == 0):
        return;
    print("We are on node " + chr(65 + l[0][0]))
    for (x,y,n) in l:
        route = ""
        if (n != 0):
            route = " using route " + str(n)
        print("Go from " + chr(65 + x) + " to " + chr(65 + y) + route)


display_path(res)

# 3. Calcul de complexite

In [ ]:
l = [i for i in range(10,19)]

def adj_mat(n):
    adj_matrix_snow = gen_random_city(n)
    adj_matrix = without_snow(adj_matrix_snow)
    return adj_matrix

def euler(adj_matrix, n):
    start = time.time()

    nodes = [str(i) for i in range(n)]
    graph = matrix_to_graph(adj_matrix, nodes)
    
    odd_nodes = get_odd_nodes(graph)
    
    all_dist = {}
    all_paths = {}
    for u in odd_nodes:
        dist, prev = dijkstra(graph, u)
        for v in odd_nodes:
            if u != v:
                all_dist.setdefault(u, {})[v] = dist[v]
                all_paths[(u, v)] = shortest_path(prev, v)
    #print("all_dist: ", all_dist)
    #print("all_paths: ", all_paths) 
    
    matching = get_min_matching(odd_nodes, all_dist)
    
    add_duplicate_edges(graph, matching, all_paths)
    
    circuit = find_eulerian_tour(graph)
    
    end = time.time()
    return end-start

t = [euler(adj_mat(i),i) for i in range(10,19)]

plt.plot(l,t)
plt.show()
